In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold


In [2]:

def clean_data(file):
    df = pd.read_csv(file)

    Y = df['ClaimNb']

    df['VehIsregular'] = (df['VehGas'] == 'Regular').astype(int)

    df = df.drop(columns= ['VehGas', 'IDpol', 'ClaimNb'])

    def letter_to_index(letter):
        return ord(letter.upper()) - ord('A')

    df['Area'] = df['Area'].apply(letter_to_index)

    df = pd.get_dummies(df)

    scaler = StandardScaler()
    X = scaler.fit_transform(df)
   
    return X, Y

def split_data(X, Y, test_size):
    return train_test_split(X, Y, test_size=test_size, random_state=1)


In [3]:
X, Y = clean_data("claims_train.csv")

X_train, X_val, y_train, y_val = split_data(X, Y, 0.2)

In [ ]:
def tune_models(X_train, X_val, y_train, y_val):
    
    results = {} 
    
    cv = KFold(n_splits=3, shuffle=True, random_state=1)

    dt_regressor = DecisionTreeRegressor(random_state=1)
    dt_params = {'ccp_alpha': [0.0, 0.00001, 0.0001, 0.001, 0.01]}
    
    dt = GridSearchCV(
        dt_regressor, 
        param_grid=dt_params, 
        cv=cv, 
        scoring='neg_mean_squared_error', 
        n_jobs=-1 
    )
    dt.fit(X_train, y_train)

    dt_best = dt.best_estimator_
    eval_results = evaluate_model(dt_best, X_val, y_val)
    
    results['DecisionTree'] = {
        'evaluation': eval_results,
        'best_params': dt.best_params_
    }

def evaluate_model(model, X_val, y_val):
    """Evaluate a regression model on validation data."""
    y_pred = model.predict(X_val)
    
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    return {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }
